In [0]:
%pip install sqlglot -q
%restart_python

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

# Read the webi_variables table (dedup: keep latest ingestion_ts per Document_Id + variable_id)
from pyspark.sql.window import Window

df_variables = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables") \
    .withColumn("_rank", F.row_number().over(
        Window.partitionBy("Document_Id", "variable_id").orderBy(F.col("ingestion_ts").desc())
    )) \
    .filter(F.col("_rank") == 1) \
    .drop("_rank") \
    .filter(F.col("variable_definition").isNotNull() & (F.col("variable_definition") != ""))

# UDF to extract datafield references from variable_definition
# Patterns: [field_name] or [provider].[field_name]
# Returns list of (provider_qualifier, field_name) tuples
def extract_datafields(definition):
    if not definition:
        return []
    
    results = []
    # Match [X].[Y] pattern (qualified) or standalone [Z] pattern
    # First find all bracket groups
    # Pattern: [provider].[field] or just [field]
    qualified_pattern = r'\[([^\]]+)\]\.\[([^\]]+)\]'
    standalone_pattern = r'\[([^\]]+)\]'
    
    # Find all qualified references first
    qualified_matches = re.findall(qualified_pattern, definition)
    qualified_fields = set()
    for provider, field in qualified_matches:
        results.append((provider, field))
        qualified_fields.add(field)
    
    # Remove qualified patterns from definition to find standalone ones
    cleaned = re.sub(qualified_pattern, '', definition)
    
    # Find standalone [field] references in remaining text
    standalone_matches = re.findall(standalone_pattern, cleaned)
    for field in standalone_matches:
        # Skip string literals and known non-field patterns
        if field not in qualified_fields:
            results.append((None, field))
    
    return results

# Define return schema
schema = ArrayType(StructType([
    StructField("provider_qualifier", StringType(), True),
    StructField("field_name", StringType(), False)
]))

extract_datafields_udf = F.udf(extract_datafields, schema)

# UDF to categorize variable_definition (logic from Variable Dictionary overview)
def categorize_definition(defn):
    if not defn or str(defn).strip() == '':
        return 'Empty'
    d = str(defn).strip().lower()

    # 1. Sum aggregate
    if re.search(r'=\s*sum\s*\(', d):
        return 'Sum Aggregate'
    # 2. Conditional — contains if(...)
    if re.search(r'\bif\s*\(', d):
        return 'Conditional (IF)'
    # 3. FX / Rate conversion
    if re.search(r'(loc\s*:\s*eur|eur\s*:\s*loc|exchange\s*rate|fx\s*rate)', d):
        return 'FX / Rate Conversion'
    # 4. Arithmetic DR/CR
    if re.search(r'\bdr\b.*\bcr\b|\bcr\b.*\bdr\b', d):
        return 'Arithmetic DR/CR'
    # 5. General arithmetic
    if re.search(r'[+\-*/]', d):
        return 'Arithmetic'
    # 6. Simple field reference
    return 'Simple Reference'

categorize_udf = F.udf(categorize_definition, StringType())

# Apply extraction, explode, and categorize
df_extracted = df_variables \
    .withColumn("datafields", extract_datafields_udf(F.col("variable_definition"))) \
    .withColumn("datafield_ref", F.explode("datafields")) \
    .withColumn("Variable_category", categorize_udf(F.col("variable_definition"))) \
    .select(
        F.col("Document_Id"),
        F.col("variable_id"),
        F.col("variable_name"),
        F.col("variable_datatype"),
        F.col("variable_qualification"),
        F.col("variable_definition"),
        F.col("Variable_category"),
        F.col("datafield_ref.provider_qualifier").alias("provider_qualifier"),
        F.col("datafield_ref.field_name").alias("extracted_datafield")
    )
df_extracted = df_extracted.dropDuplicates()
display(df_extracted.filter(F.col("Document_Id") == 412320))

In [0]:
# Read the webi_datadictionary table
from pyspark.sql import Window

df_datadict = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary") \
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("Document_Id", "Data_Provider_ID", "datafield_id").orderBy(F.col("ingestion_ts").desc())
        )
    ) \
    .filter(F.col("row_num") == 1) \
    .select(
        F.col("Document_Id"),
        F.col("Data_Provider_ID"),
        F.col("Data_Provider_Name"),
        F.col("datafield_id"),
        F.col("datafield_name"),
        F.col("datafield_qualification"),
        F.col("DataSource_Type"),
        F.col("DataSource_Name"),
        F.col("datafield_datasourceObjectId")
    )

# Join on Document_Id and extracted_datafield = datafield_name
# For qualified references [provider].[field], also match on Data_Provider_Name
df_lineage = df_extracted.join(
    df_datadict,
    on=[
        df_extracted["Document_Id"] == df_datadict["Document_Id"],
        df_extracted["extracted_datafield"] == df_datadict["datafield_name"],
        # Match provider qualifier when available, otherwise allow any provider
        (df_extracted["provider_qualifier"].isNull()) |
        (df_extracted["provider_qualifier"] == df_datadict["Data_Provider_Name"])
    ],
    how="left"
).select(
    df_extracted["Document_Id"],
    df_extracted["variable_id"],
    df_extracted["variable_name"],
    df_extracted["variable_datatype"],
    df_extracted["variable_qualification"],
    df_extracted["variable_definition"],
    df_extracted["extracted_datafield"],
    df_extracted["provider_qualifier"],
    df_datadict["Data_Provider_ID"],
    df_datadict["datafield_id"],
    df_datadict["datafield_name"],
    df_datadict["datafield_qualification"],
    df_datadict["DataSource_Type"],
    df_datadict["DataSource_Name"],
    df_datadict["datafield_datasourceObjectId"]
)
df_lineage.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
display(df_lineage.filter(F.col("Document_Id") == 412320))

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage as(
    select wbl1.*, clean_cms.Universe_ID, clean_cms.Universe_Name, clean_cms.webi_all_sql_queries from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage as wbl1 left join
    (select Document_Id, Data_Provider_ID, DataSource_Type, DataSource_ID as Universe_ID, DataSource_Name as Universe_Name, string_agg(SQL_Query, ' | ') as webi_all_sql_queries
    from (
        select *
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        qualify row_number() over (partition by Document_Id, Data_Provider_ID, SQL_Index order by ingestion_ts desc) = 1
    )
    group by Document_Id, Data_Provider_ID, DataSource_Type, DataSource_ID, DataSource_Name) as clean_cms
    on trim(wbl1.Document_Id) = trim(clean_cms.Document_Id) and trim(wbl1.Data_Provider_ID) = trim(clean_cms.Data_Provider_ID)
)

-- select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage limit 300

In [0]:
import json
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

from pyspark.sql.window import Window

df_datadictionary = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary")

# Dedup: keep only latest ingestion_ts per (Document_Id, Data_Provider_ID, datafield_id)
dedup_window = Window.partitionBy("Document_Id", "Data_Provider_ID", "datafield_id").orderBy(F.col("ingestion_ts").desc())
df_datadictionary = df_datadictionary \
    .withColumn("_rank", F.row_number().over(dedup_window)) \
    .filter(F.col("_rank") == 1) \
    .drop("_rank")

# Parse datafield_datasourcePath (JSON array like ["22","2222","22"]) into path string \22\2222\22\
def parse_path(raw):
    if not raw:
        return None
    try:
        parsed = json.loads(raw)
        # Handle {"value": ["folder1", "folder2"]} format
        if isinstance(parsed, dict) and 'value' in parsed:
            items = parsed['value']
        elif isinstance(parsed, list):
            items = parsed
        else:
            return None
        if items and isinstance(items, list):
            return '\\' + '\\'.join(items) + '\\'
        return None
    except Exception:
        return None

parse_path_udf = udf(parse_path, StringType())

df_datadictionary = df_datadictionary.withColumn("datafield_datasourcePath", parse_path_udf(F.col("datafield_datasourcePath")))

df_datadictionary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_datadictionary_temp")

display(df_datadictionary)

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage as(
with va2 as (
  select va1.*, wd1.datafield_datasourcePath
 from (select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage ) as va1
  left join(
    select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_datadictionary_temp
    qualify row_number() over (partition by Document_Id, Data_Provider_ID,datafield_id order by ingestion_ts desc) = 1  ) as wd1
  on va1.Document_Id=wd1.Document_Id and va1.Data_Provider_ID=wd1.Data_Provider_ID and va1.datafield_id=wd1.datafield_id
)
select va2.*,  wd2.description as datafield_description, wd2.sql_definition
 from va2 left join (
    select distinct universe_name,record_type, Name, Cuid, Data_Type, Description, case when SQL_Definition_Select is not null then SQL_Definition_Select else 'Select' end as SQL_Definition, Path from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_unvx_definitions
  ) as wd2
  on case when lower(va2.DataSource_Name) like '%_old_udt' then regexp_replace(va2.DataSource_Name,'_old_udt$','') else va2.DataSource_Name end=wd2.universe_name and (va2.datafield_datasourceObjectId=wd2.Cuid or (va2.datafield_name=wd2.Name and va2.datafield_datasourcepath=wd2.Path))
)


In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

# Read variables with definitions (dedup: keep latest ingestion_ts per Document_Id + variable_id)
from pyspark.sql.window import Window

df_vars = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables") \
    .withColumn("_rank", F.row_number().over(
        Window.partitionBy("Document_Id", "variable_id").orderBy(F.col("ingestion_ts").desc())
    )) \
    .filter(F.col("_rank") == 1) \
    .drop("_rank") \
    .filter(F.col("variable_definition").isNotNull() & (F.col("variable_definition") != "")) \
    .select("Document_Id", "variable_id", "variable_name", "variable_definition", "variable_qualification")

# Extract all [field] references from variable_definition
def extract_all_refs(definition):
    """Extract all bracket-enclosed references from a variable definition."""
    if not definition:
        return []
    # Match all [X] patterns (both standalone and qualified)
    all_refs = re.findall(r'\[([^\]]+)\]', definition)
    # Deduplicate while preserving order
    seen = set()
    unique = []
    for r in all_refs:
        if r not in seen:
            seen.add(r)
            unique.append(r)
    return unique

extract_refs_udf = F.udf(extract_all_refs, ArrayType(StringType()))

# Get all variable names per document (these are the potential targets)
var_names_per_doc = df_vars.select(
    F.col("Document_Id").alias("target_doc"),
    F.col("variable_id").alias("target_variable_id"),
    F.col("variable_name").alias("target_variable_name"),
    F.col("variable_qualification").alias("target_qualification")
)

# Extract references and explode
df_refs = df_vars \
    .withColumn("refs", extract_refs_udf(F.col("variable_definition"))) \
    .withColumn("referenced_name", F.explode("refs")) \
    .select(
        F.col("Document_Id"),
        F.col("variable_id").alias("source_variable_id"),
        F.col("variable_name").alias("source_variable_name"),
        F.col("variable_qualification").alias("source_qualification"),
        F.col("referenced_name")
    )

# Join: find references that match another variable_name in the SAME document
# This gives us variable → variable dependencies
var_to_var = df_refs.join(
    var_names_per_doc,
    (df_refs["Document_Id"] == var_names_per_doc["target_doc"]) &
    (df_refs["referenced_name"] == var_names_per_doc["target_variable_name"]) &
    (df_refs["source_variable_id"] != var_names_per_doc["target_variable_id"]),  # exclude self-ref
    "inner"
).select(
    df_refs["Document_Id"],
    F.col("source_variable_id"),
    F.col("source_variable_name"),
    F.col("source_qualification"),
    F.col("target_variable_id"),
    F.col("target_variable_name"),
    F.col("target_qualification")
).distinct()

# Compute dependency depth (how many levels deep)
# Level 0 = references only datafields (no variable deps)
# Level 1 = references at least one variable that itself has no variable deps
# etc.

total_vars = df_vars.count()
total_edges = var_to_var.count()
vars_with_deps = var_to_var.select("Document_Id", "source_variable_id").distinct().count()
vars_as_targets = var_to_var.select("Document_Id", "target_variable_id").distinct().count()

print(f"=== VARIABLE-TO-VARIABLE DEPENDENCY GRAPH ===")
print(f"\nTotal variables (with definitions): {total_vars}")
print(f"Total dependency edges (var → var): {total_edges}")
print(f"Variables that DEPEND on other vars: {vars_with_deps} ({100*vars_with_deps/total_vars:.1f}%)")
print(f"Variables that ARE depended upon:    {vars_as_targets} ({100*vars_as_targets/total_vars:.1f}%)")

# Dependency fan-out: how many other variables does each variable reference?
print(f"\n=== DEPENDENCY FAN-OUT (source → how many targets) ===")
var_to_var.groupBy("Document_Id", "source_variable_id", "source_variable_name") \
    .agg(F.countDistinct("target_variable_id").alias("depends_on_n_vars")) \
    .groupBy("depends_on_n_vars").agg(F.count("*").alias("variable_count")) \
    .orderBy("depends_on_n_vars").show(15)

# Save the dependency graph
var_to_var.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variable_dependency_graph")

print(f"\n✓ Saved to dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variable_dependency_graph")
print(f"\n=== SAMPLE DEPENDENCY CHAINS ===")
display(var_to_var.orderBy("Document_Id", "source_variable_name").limit(20))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load lineage table and dependency graph
lineage = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
graph = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variable_dependency_graph")

# Identify variables WITH lineage (leaf nodes that connect to datafields)
lineage_cols = ["Data_Provider_ID", "DataSource_Name", "DataSource_Type", "datafield_id",
                "datafield_name", "datafield_qualification", "datafield_datasourceObjectId",
                "Universe_ID", "Universe_Name", "datafield_datasourcePath",
                "datafield_description", "sql_definition"]

# Get distinct variable-level lineage (one variable can map to multiple datafields)
# For propagation, take the FIRST (most common) provider/source per variable
var_lineage = lineage.filter(F.col("Data_Provider_ID").isNotNull()) \
    .select("Document_Id", "variable_id", *lineage_cols) \
    .distinct()

# Variables WITHOUT lineage (candidates for propagation)
no_lineage = lineage.filter(F.col("Data_Provider_ID").isNull()) \
    .select("Document_Id", "variable_id").distinct()

total_no_lineage = no_lineage.count()
print(f"=== LINEAGE PROPAGATION VIA DEPENDENCY GRAPH ===")
print(f"\nVariables WITHOUT lineage (before propagation): {total_no_lineage}")
print(f"Variables WITH lineage (donors):                {var_lineage.select('Document_Id', 'variable_id').distinct().count()}")
print(f"Dependency graph edges:                         {graph.count()}")

# Iterative propagation: follow dependency chains
# For each unresolved variable, find its target variables in the graph
# If a target has lineage, inherit it
max_iterations = 5
resolved_all = None

# Start with variables that have direct lineage as "resolved"
resolved_vars = var_lineage.select(
    F.col("Document_Id").alias("r_doc"),
    F.col("variable_id").alias("r_var_id"),
    *[F.col(c).alias(f"r_{c}") for c in lineage_cols]
)

for iteration in range(max_iterations):
    # Find unresolved variables that depend on resolved ones
    # graph: source_variable_id depends on target_variable_id
    newly_resolved = no_lineage.alias("unresolved").join(
        graph.alias("g"),
        (F.col("unresolved.Document_Id") == F.col("g.Document_Id")) &
        (F.col("unresolved.variable_id") == F.col("g.source_variable_id")),
        "inner"
    ).join(
        resolved_vars,
        (F.col("g.Document_Id") == F.col("r_doc")) &
        (F.col("g.target_variable_id") == F.col("r_var_id")),
        "inner"
    ).select(
        F.col("unresolved.Document_Id").alias("prop_doc"),
        F.col("unresolved.variable_id").alias("prop_var_id"),
        *[F.col(f"r_{c}").alias(f"prop_{c}") for c in lineage_cols]
    ).distinct()

    # Deduplicate: if a variable depends on multiple resolved targets, pick one
    # Prefer universe-based sources over others
    newly_resolved = newly_resolved.withColumn("_rn", F.row_number().over(
        Window.partitionBy("prop_doc", "prop_var_id").orderBy(
            F.when(F.col("prop_Universe_Name").isNotNull(), 0).otherwise(1),
            F.col("prop_DataSource_Name")
        )
    )).filter(F.col("_rn") == 1).drop("_rn")

    new_count = newly_resolved.count()
    print(f"  Iteration {iteration + 1}: resolved {new_count} additional variables")

    if new_count == 0:
        break

    # Accumulate results
    if resolved_all is None:
        resolved_all = newly_resolved
    else:
        resolved_all = resolved_all.union(newly_resolved)

    # Add newly resolved to the resolved pool for next iteration
    new_resolved_vars = newly_resolved.select(
        F.col("prop_doc").alias("r_doc"),
        F.col("prop_var_id").alias("r_var_id"),
        *[F.col(f"prop_{c}").alias(f"r_{c}") for c in lineage_cols]
    )
    resolved_vars = resolved_vars.union(new_resolved_vars)

    # Remove newly resolved from unresolved pool
    no_lineage = no_lineage.join(
        newly_resolved.select(
            F.col("prop_doc").alias("Document_Id"),
            F.col("prop_var_id").alias("variable_id")
        ),
        on=["Document_Id", "variable_id"],
        how="left_anti"
    )

# Apply propagated lineage back to the main table
if resolved_all is not None:
    total_propagated = resolved_all.count()
    print(f"\n✓ Total propagated: {total_propagated} variables ({100*total_propagated/total_no_lineage:.1f}% of unresolved)")

    # Join propagated lineage back to original rows
    lineage_updated = lineage.alias("l").join(
        resolved_all.alias("p"),
        (F.col("l.Document_Id") == F.col("p.prop_doc")) &
        (F.col("l.variable_id") == F.col("p.prop_var_id")) &
        (F.col("l.Data_Provider_ID").isNull()),  # only fill where currently NULL
        "left"
    ).select(
        F.col("l.Document_Id"),
        F.col("l.variable_id"),
        F.col("l.variable_name"),
        F.col("l.variable_datatype"),
        F.col("l.variable_qualification"),
        F.col("l.variable_definition"),
        F.col("l.extracted_datafield"),
        F.col("l.provider_qualifier"),
        F.coalesce(F.col("l.Data_Provider_ID"), F.col("p.prop_Data_Provider_ID")).alias("Data_Provider_ID"),
        F.coalesce(F.col("l.datafield_id"), F.col("p.prop_datafield_id")).alias("datafield_id"),
        F.coalesce(F.col("l.datafield_name"), F.col("p.prop_datafield_name")).alias("datafield_name"),
        F.coalesce(F.col("l.datafield_qualification"), F.col("p.prop_datafield_qualification")).alias("datafield_qualification"),
        F.coalesce(F.col("l.DataSource_Type"), F.col("p.prop_DataSource_Type")).alias("DataSource_Type"),
        F.coalesce(F.col("l.DataSource_Name"), F.col("p.prop_DataSource_Name")).alias("DataSource_Name"),
        F.coalesce(F.col("l.datafield_datasourceObjectId"), F.col("p.prop_datafield_datasourceObjectId")).alias("datafield_datasourceObjectId"),
        F.coalesce(F.col("l.Universe_ID"), F.col("p.prop_Universe_ID")).alias("Universe_ID"),
        F.coalesce(F.col("l.Universe_Name"), F.col("p.prop_Universe_Name")).alias("Universe_Name"),
        F.col("l.webi_all_sql_queries"),
        F.coalesce(F.col("l.datafield_datasourcePath"), F.col("p.prop_datafield_datasourcePath")).alias("datafield_datasourcePath"),
        F.coalesce(F.col("l.datafield_description"), F.col("p.prop_datafield_description")).alias("datafield_description"),
        F.coalesce(F.col("l.sql_definition"), F.col("p.prop_sql_definition")).alias("sql_definition")
    )

    # Write updated lineage
    lineage_updated.write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")

    # Final verification
    final = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
    final_total = final.count()
    final_filled = final.filter(F.col("Data_Provider_ID").isNotNull()).count()
    print(f"\n=== FINAL AVAILABILITY RATE ===")
    print(f"Total rows:            {final_total:,}")
    print(f"With lineage:          {final_filled:,} ({100*final_filled/final_total:.1f}%)")
    print(f"Without lineage:       {final_total - final_filled:,} ({100*(final_total-final_filled)/final_total:.1f}%)")
    print(f"\nImprovement: {total_propagated} variables now have inherited lineage from their dependencies.")
else:
    print("\nNo propagation possible — no unresolved variables depend on resolved ones.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DOC = "17294606"

# ── 1. Check df_extracted (after Step 1 explode) ──────────────────────────────
# Re-run the extraction UDF inline to see what the explode produces for this doc
from pyspark.sql.types import ArrayType, StringType, StructType, StructField
import re

def extract_datafields(definition):
    if not definition:
        return []
    results = []
    qualified_pattern = r'\[([^\]]+)\]\.\[([^\]]+)\]'
    standalone_pattern = r'\[([^\]]+)\]'
    qualified_matches = re.findall(qualified_pattern, definition)
    qualified_fields = set()
    for provider, field in qualified_matches:
        results.append((provider, field))
        qualified_fields.add(field)
    cleaned = re.sub(qualified_pattern, '', definition)
    standalone_matches = re.findall(standalone_pattern, cleaned)
    for field in standalone_matches:
        if field not in qualified_fields:
            results.append((None, field))
    return results

schema = ArrayType(StructType([
    StructField("provider_qualifier", StringType(), True),
    StructField("field_name", StringType(), False)
]))
extract_datafields_udf = F.udf(extract_datafields, schema)

df_vars_raw = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables") \
    .withColumn("_rank", F.row_number().over(
        Window.partitionBy("Document_Id", "variable_id").orderBy(F.col("ingestion_ts").desc())
    )) \
    .filter((F.col("_rank") == 1) & (F.col("Document_Id") == DOC)) \
    .drop("_rank") \
    .filter(F.col("variable_definition").isNotNull() & (F.col("variable_definition") != ""))

df_extracted_doc = df_vars_raw \
    .withColumn("datafields", extract_datafields_udf(F.col("variable_definition"))) \
    .withColumn("datafield_ref", F.explode("datafields")) \
    .select(
        F.col("variable_id"),
        F.col("variable_name"),
        F.col("datafield_ref.provider_qualifier").alias("provider_qualifier"),
        F.col("datafield_ref.field_name").alias("extracted_datafield")
    ).dropDuplicates()

total_extracted = df_extracted_doc.count()
unique_vars     = df_extracted_doc.select("variable_id").distinct().count()
print(f"After Step 1 explode — doc {DOC}:")
print(f"  Rows                     : {total_extracted:,}")
print(f"  Distinct variable_ids    : {unique_vars:,}")
print(f"  Avg extracted refs/var   : {total_extracted/unique_vars:.1f}")

# ── 2. Check how many dict matches each extracted field gets (fan-out) ────────
df_dict_doc = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary") \
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("Document_Id", "Data_Provider_ID", "datafield_id")
              .orderBy(F.col("ingestion_ts").desc())
    )) \
    .filter((F.col("_rn") == 1) & (F.col("Document_Id") == DOC)) \
    .drop("_rn")

# Count how many dict rows exist per (Document_Id, datafield_name)
dict_fanout = df_dict_doc \
    .groupBy("datafield_name") \
    .agg(
        F.countDistinct("Data_Provider_ID").alias("num_providers"),
        F.countDistinct("datafield_id").alias("num_datafield_ids"),
        F.count("*").alias("total_dict_rows")
    ) \
    .filter(F.col("total_dict_rows") > 1) \
    .orderBy(F.col("total_dict_rows").desc())

print(f"\nDict fields with >1 row per name in doc {DOC} (fan-out risk):")
display(dict_fanout.limit(20))

# ── 3. Simulate Step 2 join for this document — count rows per variable ───────
df_lineage_doc = df_extracted_doc.join(
    df_dict_doc,
    on=[
        df_extracted_doc["extracted_datafield"] == df_dict_doc["datafield_name"],
        (df_extracted_doc["provider_qualifier"].isNull()) |
        (df_extracted_doc["provider_qualifier"] == df_dict_doc["Data_Provider_Name"])
    ],
    how="left"
).select(
    df_extracted_doc["variable_id"],
    df_extracted_doc["extracted_datafield"],
    df_extracted_doc["provider_qualifier"],
    df_dict_doc["Data_Provider_ID"],
    df_dict_doc["datafield_id"],
    df_dict_doc["datafield_name"]
)

post_join_count = df_lineage_doc.count()
print(f"\nAfter Step 2 join — doc {DOC}:")
print(f"  Rows                                   : {post_join_count:,}")
print(f"  Fan-out vs extracted rows              : {post_join_count - total_extracted:,} extra rows")

# Show variables with the most rows after join
print("\nTop variables by row count after join:")
display(
    df_lineage_doc
    .groupBy("variable_id")
    .agg(
        F.count("*").alias("rows_after_join"),
        F.countDistinct(F.struct("extracted_datafield", "Data_Provider_ID", "datafield_id")).alias("distinct_combinations")
    )
    .withColumn("fan_out", F.col("rows_after_join") > F.col("distinct_combinations"))
    .orderBy(F.col("rows_after_join").desc())
    .limit(20)
)

# ── 4. Zoom in on the specific variable causing duplicates ────────────────────
print("\nStep 2 join result for a high-duplicate variable (first by row count):")
top_var = df_lineage_doc.groupBy("variable_id").count().orderBy(F.col("count").desc()).first()["variable_id"]
print(f"  Showing variable_id = {top_var}")
display(
    df_lineage_doc.filter(F.col("variable_id") == top_var)
    .orderBy("extracted_datafield", "Data_Provider_ID")
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

lineage = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")

before = lineage.count()
before_vars = lineage.select("Document_Id", "variable_id").distinct().count()
before_keys = lineage.select("Document_Id", "variable_id", "datafield_id").distinct().count()

print(f"Before dedup:")
print(f"  Total rows                          : {before:,}")
print(f"  Distinct (Document_Id, variable_id) : {before_vars:,}")
print(f"  Distinct (Document_Id, var, dict)   : {before_keys:,}")
print(f"  Rows with same (var, dict_item)     : {before - before_keys:,}  ← target for removal")

# ── Deduplication strategy ────────────────────────────────────────────────────────────────────────────────────
# Root cause: after Step 6 propagation, a variable with N references to
# other variables can have all N resolved to the same dict item via the
# dependency graph. This produces N rows per (variable_id, datafield_id).
#
# Fix: keep ONE row per (Document_Id, variable_id, datafield_id).
# Priority within each group:
#   1. Direct dict match (extracted_datafield == datafield_name) — most reliable
#   2. Row where extracted_datafield is NOT a "*" prefixed variable reference
#   3. Any remaining (propagated)
lineage_deduped = lineage.withColumn(
    "_priority",
    F.when(
        F.col("extracted_datafield") == F.col("datafield_name"),
        1  # Direct: extracted field name matches the dict field name exactly
    ).when(
        ~F.col("extracted_datafield").startswith("*"),
        2  # Likely a real field reference (no * prefix)
    ).otherwise(
        3  # Propagated via a variable reference
    )
).withColumn(
    "_rn",
    F.row_number().over(
        Window.partitionBy("Document_Id", "variable_id", "datafield_id")
              .orderBy(
                  F.col("_priority"),
                  F.col("datafield_id")   # tie-break deterministically
              )
    )
).filter(F.col("_rn") == 1).drop("_rn", "_priority")

after = lineage_deduped.count()
after_vars = lineage_deduped.select("Document_Id", "variable_id").distinct().count()
removed = before - after

print(f"\nAfter dedup:")
print(f"  Total rows                          : {after:,}")
print(f"  Distinct (Document_Id, variable_id) : {after_vars:,}  (unchanged ✓)")
print(f"  Rows removed                        : {removed:,}")
print(f"  (Document_Id, variable_id, dict)    : now unique ✓")

# ── Spot-check doc 17294606 ───────────────────────────────────────────────────────────────────────────────────
spot = lineage_deduped.filter(F.col("Document_Id") == "17294606")
spot_vars  = spot.select("variable_id").distinct().count()
spot_rows  = spot.count()
spot_refs  = spot.filter(F.col("datafield_id").isNotNull()).select("datafield_id").distinct().count()
print(f"\nSpot-check doc 17294606:")
print(f"  Variables          : {spot_vars:,}")
print(f"  Total rows         : {spot_rows:,}")
print(f"  Distinct dict refs : {spot_refs:,}")
print(f"  Avg dict items/var : {spot_rows/spot_vars:.2f}")

# ── Write back ────────────────────────────────────────────────────────────────────────────────────
target = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage"
lineage_deduped.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
print(f"\n✓ Written {after:,} rows to {target}")
print(f"  {removed:,} duplicate (variable, dict_item) rows removed.")
print(f"  Re-run KPI_V2 cell 10 to rebuild webi_data_entries with clean data.")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Load sources ──────────────────────────────────────────────────────────────────
lineage = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
graph   = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variable_dependency_graph")
raw_dict = (
    spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary")
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("Document_Id", "Data_Provider_ID", "datafield_id")
              .orderBy(F.col("ingestion_ts").desc())
    ))
    .filter(F.col("_rn") == 1).drop("_rn")
    .select("Document_Id", "datafield_name", "Data_Provider_ID")
)

# ── 1. Overall linkage summary ──────────────────────────────────────────────────────
total_rows  = lineage.count()
total_vars  = lineage.select("Document_Id", "variable_id").distinct().count()
rows_linked = lineage.filter(F.col("datafield_id").isNotNull()).count()
rows_null   = lineage.filter(F.col("datafield_id").isNull()).count()

vars_fully_linked = (
    lineage.groupBy("Document_Id", "variable_id")
    .agg(F.max(F.col("datafield_id").isNotNull().cast("int")).alias("has_any_link"))
    .filter(F.col("has_any_link") == 1).count()
)
vars_no_link = total_vars - vars_fully_linked

print("=" * 65)
print("OVERALL LINKAGE SUMMARY")
print(f"  Total rows in webi_variables_linage : {total_rows:,}")
print(f"  Rows linked to a dict item          : {rows_linked:,} ({100*rows_linked/total_rows:.1f}%)")
print(f"  Rows with NULL dict linkage         : {rows_null:,} ({100*rows_null/total_rows:.1f}%)")
print(f"  Variables with at least 1 dict link : {vars_fully_linked:,} / {total_vars:,} ({100*vars_fully_linked/total_vars:.1f}%)")
print(f"  Variables with NO dict link at all  : {vars_no_link:,} ({100*vars_no_link/total_vars:.1f}%)")

# ── 2. Classify each NULL row by WHY it couldn’t be resolved ─────────────────────
# Build lookup tables:
#   a) All (Document_Id, variable_name) — to detect var-ref extracted fields
#   b) All (Document_Id, datafield_name) from raw dict — to detect dict-name mismatches
#   c) All (Document_Id, variable_id) with resolved lineage — to detect dead-end var-refs

all_var_names = lineage.select(
    F.col("Document_Id").alias("vn_doc"),
    F.col("variable_name").alias("vn_name"),
    F.col("variable_id").alias("vn_id")
).distinct()

resolved_var_ids = (
    lineage.filter(F.col("datafield_id").isNotNull())
    .select(F.col("Document_Id").alias("rv_doc"),
            F.col("variable_id").alias("rv_id"))
    .distinct()
)

dict_field_names = raw_dict.select(
    F.col("Document_Id").alias("dn_doc"),
    F.lower(F.trim(F.col("datafield_name"))).alias("dn_name")
).distinct()

null_rows = lineage.filter(F.col("datafield_id").isNull())

# Step A: is extracted_datafield a known variable name in the same document?
null_rows = null_rows.join(
    all_var_names,
    (null_rows["Document_Id"] == all_var_names["vn_doc"]) &
    (null_rows["extracted_datafield"] == all_var_names["vn_name"]),
    "left"
).select(null_rows["*"],
         F.col("vn_id").isNotNull().alias("is_var_ref"),
         F.col("vn_id").alias("target_var_id"))

# Step B: if it IS a var-ref, is the target variable itself resolved?
null_rows = null_rows.join(
    resolved_var_ids,
    (null_rows["Document_Id"] == resolved_var_ids["rv_doc"]) &
    (null_rows["target_var_id"] == resolved_var_ids["rv_id"]),
    "left"
).select(null_rows["*"],
         F.col("rv_id").isNotNull().alias("target_is_resolved"))

# Step C: is extracted_datafield present in the raw dict (case-insensitive check)?
null_rows = null_rows.join(
    dict_field_names,
    (null_rows["Document_Id"] == dict_field_names["dn_doc"]) &
    (F.lower(F.trim(null_rows["extracted_datafield"])) == dict_field_names["dn_name"]),
    "left"
).select(null_rows["*"],
         F.col("dn_name").isNotNull().alias("exists_in_raw_dict"))

# Classify
null_classified = null_rows.withColumn(
    "unresolved_reason",
    F.when(
        F.col("extracted_datafield").isNull() | (F.trim(F.col("extracted_datafield")) == ""),
        "A: BLANK extracted_datafield"
    ).when(
        F.col("is_var_ref") & F.col("target_is_resolved"),
        "B: VAR_REF — target resolved but propagation missed"
    ).when(
        F.col("is_var_ref") & ~F.col("target_is_resolved"),
        "C: VAR_REF — target variable also has no dict link (dead-end chain)"
    ).when(
        ~F.col("is_var_ref") & F.col("exists_in_raw_dict"),
        "D: DICT NAME EXISTS — join condition blocked match (provider filter?)"
    ).otherwise(
        "E: UNKNOWN FIELD — not in dict or variables for this doc"
    )
)

# ── 3. Summary by reason ─────────────────────────────────────────────────────────
reason_summary = (
    null_classified
    .groupBy("unresolved_reason")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("Document_Id", "variable_id").alias("distinct_variables"),
        F.countDistinct("Document_Id").alias("distinct_docs")
    )
    .orderBy("unresolved_reason")
)
print("\n" + "=" * 65)
print("NULL DICT LINKAGE — ROOT CAUSE BREAKDOWN")
display(reason_summary)

# ── 4. Sample rows for each reason category ───────────────────────────────────
for reason in [
    "B: VAR_REF — target resolved but propagation missed",
    "C: VAR_REF — target variable also has no dict link (dead-end chain)",
    "D: DICT NAME EXISTS — join condition blocked match (provider filter?)",
    "E: UNKNOWN FIELD — not in dict or variables for this doc",
]:
    subset = null_classified.filter(F.col("unresolved_reason") == reason)
    cnt = subset.count()
    if cnt == 0:
        continue
    print(f"\n{'='*65}")
    print(f"{reason}  ({cnt:,} rows)")
    display(
        subset.select(
            "Document_Id", "variable_id", "variable_name",
            "extracted_datafield", "variable_definition"
        ).limit(5)
    )

# ── 5. Depth check: how deep is the dead-end chain (reason C)? ─────────────────
# Check if dead-end target variables appear anywhere as a source in the dep graph
# (i.e., they have further dependencies that might eventually resolve)
dead_end_targets = (
    null_classified
    .filter(F.col("unresolved_reason") == "C: VAR_REF — target variable also has no dict link (dead-end chain)")
    .select(
        F.col("Document_Id"),
        F.col("target_var_id").alias("variable_id")
    ).distinct()
)

# Do dead-end targets have outgoing edges in the dep graph?
has_further_deps = dead_end_targets.join(
    graph.select(F.col("Document_Id"), F.col("source_variable_id")),
    (dead_end_targets["Document_Id"] == graph["Document_Id"]) &
    (dead_end_targets["variable_id"] == graph["source_variable_id"]),
    "left"
).select(
    dead_end_targets["*"],
    F.col("source_variable_id").isNotNull().alias("has_dep_graph_edge")
).distinct()

dead_total  = has_further_deps.count()
dead_deeper = has_further_deps.filter(F.col("has_dep_graph_edge")).count()
dead_leaf   = dead_total - dead_deeper

print("\n" + "=" * 65)
print("DEAD-END CHAIN DEPTH ANALYSIS (Reason C targets)")
print(f"  Dead-end target variables total            : {dead_total:,}")
print(f"  Have further dep graph edges (deeper chain): {dead_deeper:,}  ← exceeded 5-iteration limit")
print(f"  True leaf — no deps at all in graph        : {dead_leaf:,}  ← genuinely unresolvable")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Reload sources ────────────────────────────────────────────────────────────────
lineage = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_variables_linage")
raw_dict = (
    spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary")
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("Document_Id", "Data_Provider_ID", "datafield_id")
              .orderBy(F.col("ingestion_ts").desc())
    ))
    .filter(F.col("_rn") == 1).drop("_rn")
    .select("Document_Id", "Data_Provider_ID", "Data_Provider_Name", "datafield_name")
)

# ── Isolate the 8 Reason-D rows ─────────────────────────────────────────────────────────
# Reason D: field name IS in the dictionary for that doc (case-insensitive)
# but the Step 2 join rejected the row because provider_qualifier != Data_Provider_Name
all_var_names = lineage.select(
    F.col("Document_Id").alias("vn_doc"),
    F.col("variable_name").alias("vn_name")
).distinct()

dict_field_names = raw_dict.select(
    F.col("Document_Id").alias("dn_doc"),
    F.lower(F.trim(F.col("datafield_name"))).alias("dn_name")
).distinct()

null_rows = lineage.filter(F.col("datafield_id").isNull())

# Flag: is extracted_datafield a variable name?
null_rows = null_rows.join(
    all_var_names,
    (null_rows["Document_Id"] == all_var_names["vn_doc"]) &
    (null_rows["extracted_datafield"] == all_var_names["vn_name"]),
    "left"
).select(null_rows["*"], F.col("vn_name").isNotNull().alias("is_var_ref"))

# Flag: does field name exist in raw dict (case-insensitive)?
null_rows = null_rows.join(
    dict_field_names,
    (null_rows["Document_Id"] == dict_field_names["dn_doc"]) &
    (F.lower(F.trim(null_rows["extracted_datafield"])) == dict_field_names["dn_name"]),
    "left"
).select(null_rows["*"], F.col("dn_name").isNotNull().alias("exists_in_raw_dict"))

reason_d = null_rows.filter(
    ~F.col("is_var_ref") & F.col("exists_in_raw_dict")
).select(
    "Document_Id", "variable_id", "variable_name",
    "extracted_datafield", "provider_qualifier", "variable_definition"
)

print(f"Reason D rows: {reason_d.count()}")
print("\nAll Reason D rows — what the formula extracted:")
display(reason_d)

# ── For each, show what Data_Provider_Name actually exists in the dict ────────────────
# This reveals what provider_qualifier should have matched
actual_providers = reason_d.join(
    raw_dict,
    (reason_d["Document_Id"] == raw_dict["Document_Id"]) &
    (F.lower(F.trim(reason_d["extracted_datafield"])) == F.lower(F.trim(raw_dict["datafield_name"]))),
    "inner"
).select(
    reason_d["Document_Id"],
    reason_d["variable_id"],
    reason_d["variable_name"],
    reason_d["extracted_datafield"],
    reason_d["provider_qualifier"].alias("formula_provider_qualifier"),
    raw_dict["Data_Provider_ID"].alias("actual_provider_id"),
    raw_dict["Data_Provider_Name"].alias("actual_provider_name"),
    reason_d["variable_definition"]
).orderBy("Document_Id", "variable_id")

print("\nMismatch detail — formula qualifier vs actual dict provider name:")
display(actual_providers)

# ── Root cause summary ──────────────────────────────────────────────────────────────────
# Classify each mismatch type
mismatch_types = actual_providers.withColumn(
    "mismatch_type",
    F.when(
        F.col("formula_provider_qualifier").isNull(),
        "NULL qualifier — should have matched any provider (join logic issue)"
    ).when(
        F.lower(F.trim(F.col("formula_provider_qualifier"))) ==
        F.lower(F.trim(F.col("actual_provider_name"))),
        "CASE/SPACE mismatch — same string but different casing or whitespace"
    ).otherwise(
        "NAME MISMATCH — qualifier uses display name, dict uses internal provider name"
    )
)

print("\nMismatch type classification:")
display(
    mismatch_types.groupBy("mismatch_type")
    .agg(
        F.count("*").alias("rows"),
        F.collect_set(F.struct(
            F.col("formula_provider_qualifier"),
            F.col("actual_provider_name")
        )).alias("qualifier_vs_actual_examples")
    )
)
